# 08d. Supplementary checks

Reads caches and results of notebooks 01–08c (no audio decoding). Section F needs the embedding caches of notebook 07.

- A. D1: leak vs. false alarm among flagged records
- B. D1: sensor-ID lookup at unseen sites (Table 3)
- C. D1: majority rate of E10/E11
- D. D3: LOTO vs. run-independent split with matched training size
- E. D2: metadata inside 03:30–04:30
- F. D2: acoustic vs. clock inside 03:30–04:30, by recording format

In [2]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import src as la
from src import config as cfg, io_utils, evaluation as ev, sfb, diagnostics as dg, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import warnings; warnings.filterwarnings("ignore")
CAP = cfg.D1_SUBSAMPLE

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## A. D1: leak vs. false alarm (flagged records)

- Rows: out, in, noise, other (all `lrate` = 90); leak = out, in; false alarm = noise, other
- Inputs: spectrum (513 bins) / `llevel`
- Folds: stratified, sensor-grouped, site-grouped (5 folds each); GBDT, training rows capped at 30,000
- Intervals: site-level bootstrap

In [3]:
X1, X1_max, M1, HZ = io_utils.load_d1()
y5 = M1.cls.to_numpy(); site = M1.site.to_numpy(); sid = M1.sid.to_numpy()
flag = np.isin(y5, ["out", "in", "noise", "other"])
yf = np.where(np.isin(y5[flag], ["out", "in"]), "leak", "false_alarm")
print(f"flagged records: {flag.sum():,}; leak share = {np.mean(yf == 'leak'):.4f}; "
      f"lrate values on these rows: {sorted(M1.lrate[flag].unique())}")
parts = []
for pname, scheme, grp in [("naive (stratified 5-fold)", "stratified", None),
                           ("sensor-grouped 5-fold", "stratified_group", sid[flag]),
                           ("site-grouped 5-fold", "stratified_group", site[flag])]:
    folds = ev.make_folds(scheme, yf, grp, 5, seed=0)
    summ, _ = dg.incremental_value({"acoustic: 513 bins": X1[flag],
                                    "metadata: llevel": M1.llevel.to_numpy(float)[flag].reshape(-1, 1)},
                                   yf, folds, kind="gbdt", seed=0, clusters=site[flag],
                                   subsample=CAP, n_boot=cfg.N_BOOT)
    rec = []
    for c_ in summ.cm:
        d = json.loads(c_); m = np.asarray(d["m"])
        rec.append(dict(zip([f"recall_{l}" for l in d["labels"]], (m.diagonal() / m.sum(1)).round(4))))
    summ = pd.concat([summ.reset_index(drop=True), pd.DataFrame(rec)], axis=1)
    summ.insert(0, "protocol", pname)
    parts.append(summ)
fa = pd.concat(parts, ignore_index=True)
fa.insert(0, "task", "flagged records: leak (out, in) vs false alarm (noise, other)")
fa.to_csv(cfg.RESULTS / "S_A_D1_flagged_leak_vs_false_alarm.csv", index=False)
print(fa.drop(columns=["cm", "task"]).round(4).to_string(index=False))
for _, r in fa.iterrows():
    rp.record(f"D1.flagged.{r.protocol.split(' ')[0]}.{r.feature_set.split(':')[0]}.acc", r.acc,
              "S_A_D1_flagged_leak_vs_false_alarm.csv", "R2-3,R2-4", lo=r.acc_lo, hi=r.acc_hi,
              note=f"bacc={r.bacc:.4f}; rule={r.fold_majority_rule:.4f}; recall false alarm="
                   f"{r.get('recall_false_alarm', np.nan):.3f}; unit=site")

flagged records: 48,218; leak share = 0.7189; lrate values on these rows: [np.int64(90)]
                 protocol        feature_set     n  n_features    acc   bacc    mf1  fold_majority_rule  excess_over_rule  acc_lo  acc_hi  excess_lo  excess_hi  n_clusters  recall_false_alarm  recall_leak
naive (stratified 5-fold) acoustic: 513 bins 48218         513 0.9079 0.8600 0.8794              0.7189            0.1890  0.8831  0.9328     0.1182     0.2676          49              0.7505       0.9695
naive (stratified 5-fold)   metadata: llevel 48218           1 0.7206 0.5082 0.4404              0.7189            0.0017  0.6213  0.8095    -0.0031     0.0068          49              0.0231       0.9934
    sensor-grouped 5-fold acoustic: 513 bins 48218         513 0.7000 0.5616 0.5616              0.7189           -0.0189  0.6211  0.7724    -0.0481     0.0121          49              0.2457       0.8776
    sensor-grouped 5-fold   metadata: llevel 48218           1 0.7195 0.5059 0.4355        

## B. D1: sensor-ID lookup at unseen sites

Site-grouped folds (GroupKFold, 5) → all test sensors unseen → lookup = training-fold majority class
Per fold: fallback class, accuracy, class mix of held-out sites

In [4]:
rows = []
for k, (tr, te) in enumerate(GroupKFold(5).split(y5, y5, site)):
    fb = pd.Series(y5[tr]).value_counts().idxmax()
    mix = pd.Series(y5[te]).value_counts(normalize=True)
    rows.append(dict(fold=k, n_test=len(te), n_test_sites=len(set(site[te])), fallback_class=fb,
                     fallback_acc=float(np.mean(y5[te] == fb)), test_majority_class=mix.idxmax(),
                     test_majority_share=float(mix.max()),
                     test_mix="; ".join(f"{c}={v:.2f}" for c, v in mix.items())))
fbk = pd.DataFrame(rows)
fbk.to_csv(cfg.RESULTS / "S_B_D1_sensor_id_fallback.csv", index=False)
print(fbk.to_string(index=False))
pooled = float(np.average(fbk.fallback_acc, weights=fbk.n_test))
print(f"\npooled fallback accuracy = {pooled:.4f} (Table 3 'sensor id, site-independent' is slightly higher "
      "because a few sensor ids occur at more than one site and are therefore not unseen)")
per_site = M1.groupby("site").cls.agg(lambda s: s.value_counts(normalize=True).max())
print(f"sites whose records are all one class: {(per_site == 1).sum()} of {len(per_site)}")
rp.record("D1.SFB.sensor_id_site_indep.fallback_pooled", pooled, "S_B_D1_sensor_id_fallback.csv",
          "R1-M1,R1-minor(Table3)", note="training-majority fallback; depends on the fold assignment of sites")

 fold  n_test  n_test_sites fallback_class  fallback_acc test_majority_class  test_majority_share                                               test_mix
    0   14075             8            out      0.099538              normal             0.773783 normal=0.77; out=0.10; in=0.06; noise=0.04; other=0.02
    1   14075             8         normal      0.115382                 out             0.442984 out=0.44; in=0.33; normal=0.12; other=0.09; noise=0.03
    2   14085            10         normal      0.081718                 out             0.366915 out=0.37; in=0.28; other=0.16; noise=0.11; normal=0.08
    3   14075            14            out      0.114458              normal             0.599716 normal=0.60; other=0.18; out=0.11; in=0.09; noise=0.01
    4   14074             9         normal      0.004192                 out             0.377718 out=0.38; in=0.29; noise=0.21; other=0.11; normal=0.00

pooled fallback accuracy = 0.0831 (Table 3 'sensor id, site-independent' is sligh

## C. D1: majority rate of E10/E11

Per seed (site split, 30% test sites): outdoor share and majority rate; baseline = mean majority rate

In [5]:
mask = np.isin(y5, ["out", "in"])
yoi, soi = y5[mask], site[mask]
bt = []
for seed in range(cfg.N_SEEDS_AUX):
    a, b = next(GroupShuffleSplit(1, test_size=.3, random_state=seed).split(np.arange(len(yoi)), yoi, groups=soi))
    sh = pd.Series(yoi[b]).value_counts(normalize=True)
    bt.append(dict(seed=seed, n_test=len(b), outdoor_share=float(sh.get("out", 0)),
                   majority_class=sh.idxmax(), majority_rate=float(sh.max())))
bt = pd.DataFrame(bt)
print(bt.round(4).to_string(index=False))
print(f"\nmean majority rate = {bt.majority_rate.mean():.4f}   (the 0.560 baseline)")
print(f"mean outdoor share = {bt.outdoor_share.mean():.4f}   (not the baseline: indoor is the majority in some seeds)")
bt.to_csv(cfg.RESULTS / "S_C_D1_E10_baseline.csv", index=False)
rp.record("D1.E10.majority_mean_testfold", bt.majority_rate.mean(), "S_C_D1_E10_baseline.csv", "R1-M9",
          note=f"mean of the per-seed majority rate; mean outdoor share={bt.outdoor_share.mean():.4f}")

 seed  n_test  outdoor_share majority_class  majority_rate
    0   16554         0.5793            out         0.5793
    1   16513         0.5655            out         0.5655
    2    7586         0.4077             in         0.5923
    3   11282         0.5290            out         0.5290
    4    9978         0.5347            out         0.5347

mean majority rate = 0.5601   (the 0.560 baseline)
mean outdoor share = 0.5232   (not the baseline: indoor is the majority in some seeds)


## D. D3: LOTO vs. matched training size

- LOTO: one topology in training (30 runs: 24 leak, 6 no-leak; 1,800 segments)
- Control: run-independent 5-fold, training runs subsampled to 24 leak + 6 no-leak
- 10 repetitions (fold seed and subsample); GBDT

In [6]:
M3 = io_utils.scan_d3()
H = M3[(~M3.is_bg) & (M3.modality == "hydrophone") & M3.label.notna()].reset_index(drop=True)
c = io_utils.load_cache(f"d3_seg_hydrophone_{cfg.WIN_SEC}_pcm32", H.file_id, by_unique=True)
assert c is not None, "run notebook 06 first"
S = H.set_index("file_id").loc[c["ids"].astype(str)].reset_index()
X3, y3 = c["X"], S.label.to_numpy().astype(str)
run, topo = S.run_id.to_numpy(), S.topology.to_numpy()
runs = S.drop_duplicates("run_id")[["run_id", "label", "topology"]]
loto = ev.make_folds("logo", y3, topo)
n_leak_runs = int(runs[(runs.topology == runs.topology.iloc[0]) & (runs.label == "leak")].shape[0])
n_nl_runs = int(runs[(runs.topology == runs.topology.iloc[0]) & (runs.label == "noleak")].shape[0])
pl, _ = ev.oof_predict(X3, y3, loto, "gbdt", 0)
s_loto = ev.score(y3, pl)
rows = [dict(design="LOTO (one topology in training)", rep=0, n_train=len(loto[0][0]),
             acc=s_loto["acc"], bacc=s_loto["bacc"], rec_noleak=s_loto["rec_noleak"])]
for rep in range(cfg.N_SEEDS):
    folds = ev.make_folds("stratified_group", y3, run, 5, seed=rep)
    rng = np.random.RandomState(rep)
    pred = np.empty(len(y3), dtype=object); ntr = []
    for tr, te in folds:
        tr_runs = runs[runs.run_id.isin(set(run[tr]))]
        keep = list(rng.choice(tr_runs[tr_runs.label == "leak"].run_id, n_leak_runs, replace=False)) + \
               list(rng.choice(tr_runs[tr_runs.label == "noleak"].run_id, n_nl_runs, replace=False))
        a = tr[np.isin(run[tr], keep)]; ntr.append(len(a))
        pred[te] = ev.make_model("gbdt", rep).fit(X3[a], y3[a]).predict(X3[te])
    s = ev.score(y3, pred.astype(str))
    rows.append(dict(design="run-independent, training matched to LOTO size", rep=rep, n_train=int(np.mean(ntr)),
                     acc=s["acc"], bacc=s["bacc"], rec_noleak=s["rec_noleak"]))
md = pd.DataFrame(rows)
md.to_csv(cfg.RESULTS / "S_D_D3_LOTO_matched_size.csv", index=False)
ms = md.groupby("design").agg(n_train=("n_train", "first"), acc=("acc", "mean"), acc_sd=("acc", "std"),
                              bacc=("bacc", "mean"), bacc_min=("bacc", "min"), bacc_max=("bacc", "max"),
                              rec_noleak=("rec_noleak", "mean"), n=("acc", "size")).reset_index()
print(ms.round(4).to_string(index=False))
ctl = ms[ms.design.str.startswith("run")].iloc[0]
rp.record("D3.LOTO_matched_size.control.acc", ctl.acc, "S_D_D3_LOTO_matched_size.csv", "R2-2,R2-8",
          note=f"bacc={ctl.bacc:.4f} (min {ctl.bacc_min:.4f}); n_train={int(ctl.n_train)}; reps={int(ctl.n)}")
rp.record("D3.LOTO_matched_size.loto.acc", s_loto["acc"], "S_D_D3_LOTO_matched_size.csv", "R2-2,R2-8",
          note=f"bacc={s_loto['bacc']:.4f}")

                                        design  n_train    acc  acc_sd   bacc  bacc_min  bacc_max  rec_noleak  n
               LOTO (one topology in training)     1800 0.8025     NaN 0.5115    0.5115    0.5115      0.0264  1
run-independent, training matched to LOTO size     1800 0.8883  0.0178 0.7650    0.7030    0.8595      0.5594 10


## E. D2: metadata inside 03:30–04:30

- Fields: sensor type, file naming, pipe material, WAV header (sample rate + duration)
- Logger-independent lookup, Cramér's V
- Loggers per class

In [7]:
M2 = io_utils.scan_d2(audio_info=True)
W = M2[(M2.ext == ".wav") & M2.label.isin(["leak", "noleak"])].reset_index(drop=True)
W["logger"] = W.logger.fillna("NA")
E = W[W.timed & (W.hour >= 3.5) & (W.hour <= 4.5)].reset_index(drop=True)
if "sr" in E.columns:
    E["header"] = E.sr.astype(str) + "_" + E.duration_s.astype(float).round(0).astype(str)
ye = E.label.to_numpy()
print(pd.crosstab([E.sensor, E.naming], E.label).to_string())
print("\nindependent loggers per class in the window:")
lg = E.groupby(["sensor", "label"]).logger.nunique().unstack(fill_value=0)
print(lg.to_string())
rows = []
for col in [c_ for c_ in ["sensor", "naming", "material", "header"] if c_ in E.columns]:
    _, pred = sfb.sfb_categorical(E[col].astype(str), ye, groups=E.logger, return_pred=True)
    _, dep = sfb.metadata_dependence(E.assign(**{col: E[col].astype(str)}), col, ye)
    rows.append(dict(field=col, n=len(E), sfb_logger_indep=float(np.mean(pred.astype(str) == ye)),
                     majority=sfb.majority_baseline(ye), cramers_v=dep["cramers_v"],
                     levels=dep["levels"]))
eh = pd.DataFrame(rows)
eh.to_csv(cfg.RESULTS / "S_E_D2_early_window_metadata.csv", index=False)
print()
print(eh.round(4).to_string(index=False))
n_nl_loggers = int(E[E.label == "noleak"].logger.nunique())
n_l_loggers = int(E[E.label == "leak"].logger.nunique())
print(f"\nno-leak loggers in the window: {n_nl_loggers}; leak loggers: {n_l_loggers}")
for _, r in eh.iterrows():
    rp.record(f"D2.early_window.SFB.{r.field}", r.sfb_logger_indep, "S_E_D2_early_window_metadata.csv", "R2-6",
              note=f"majority={r.majority:.4f}; cramers_v={r.cramers_v:.3f}")
rp.record("D2.early_window.n_noleak_loggers", n_nl_loggers, "S_E_D2_early_window_metadata.csv", "R2-6",
          note=f"leak loggers={n_l_loggers}")

label                           leak  noleak
sensor        naming                        
Hydrophones   hydro_GSNNHHMM      40      20
Noise Loggers logger_date_hhmm     8       2

independent loggers per class in the window:
label          leak  noleak
sensor                     
Hydrophones      10       4
Noise Loggers     4       1

   field  n  sfb_logger_indep  majority  cramers_v  levels
  sensor 70            0.6857    0.6857     0.1005       2
  naming 70            0.6857    0.6857     0.1005       2
material 70            0.6857    0.6857     0.2392       3
  header 70            0.6000    0.6857     0.5348       2

no-leak loggers in the window: 5; leak loggers: 14


## F. D2: acoustic vs. clock by recording format (03:30–04:30)

- Per format with both classes and ≥ 2 loggers per class
- Clock (depth-1 stump on hour) vs. each embedding (logistic probe), StratifiedGroupKFold by logger, 10 fold seeds
- Differences vs. clock: paired logger-level bootstrap (fold seed 0)

In [8]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier

win = (W.timed & (W.hour >= 3.5) & (W.hour <= 4.5)).to_numpy()
Ew = W[win].copy()
Ew["header"] = Ew.sr.astype(str) + " Hz, " + Ew.duration_s.astype(float).round(0).astype(int).astype(str) + " s"
print("files per recording format, class and sensor:")
print(pd.crosstab([Ew.header, Ew.sensor], Ew.label).to_string())
print("\nloggers per recording format and class:")
print(Ew.groupby(["header", "label"]).logger.nunique().unstack(fill_value=0).to_string())

EMB = {}
for n in ["Mel2023", "WAV2VEC2_BASE", "HUBERT_BASE", "AST"]:
    c_ = io_utils.load_cache(f"d2_emb_{n}", W.file_id)
    if c_ is not None:
        EMB[n] = c_["X"][win]
print("\nembeddings found:", list(EMB))

class _Clock:
    def __init__(self): self.m = DecisionTreeClassifier(max_depth=1, random_state=0)
    def fit(self, X, y): self.m.fit(X, y); return self
    def predict(self, X): return self.m.predict(X)

def _oof(Xs, yy, folds, make):
    p = np.empty(len(yy), dtype=object)
    for tr, te in folds:
        if len(set(yy[tr])) < 2:
            p[te] = yy[tr][0]
            continue
        p[te] = make().fit(Xs[tr], yy[tr]).predict(Xs[te])
    return p.astype(str)

hdr = Ew.header.to_numpy()
rows, diffs = [], []
for h in sorted(set(hdr)):
    sel = np.where(hdr == h)[0]
    g = Ew.iloc[sel]
    yy, gg = g.label.to_numpy(), g.logger.to_numpy()
    n_lg = g.groupby("label").logger.nunique()
    base = dict(header=h, n=len(g), n_leak=int((yy == "leak").sum()), n_noleak=int((yy == "noleak").sum()),
                leak_loggers=int(n_lg.get("leak", 0)), noleak_loggers=int(n_lg.get("noleak", 0)))
    k = int(min(5, n_lg.get("leak", 0), n_lg.get("noleak", 0)))
    if k < 2:
        rows.append(dict(base, predictor="-", note="a class has fewer than 2 loggers: no logger-grouped test"))
        continue
    hour = g.hour.to_numpy().reshape(-1, 1)
    sets = {"clock (hour stump)": (hour, _Clock)}
    for n, Xe in EMB.items():
        sets[n] = (Xe[sel], (lambda s=0: ev.make_model("logreg", s)))
    res, store = [], {}
    for fs in range(cfg.N_SEEDS):
        folds = list(StratifiedGroupKFold(k, shuffle=True, random_state=fs).split(hour, yy, gg))
        for name, (Xs, make) in sets.items():
            p = _oof(Xs, yy, folds, make)
            s = ev.score(yy, p)
            res.append(dict(predictor=name, acc=s["acc"], bacc=s["bacc"]))
            if fs == 0:
                store[name] = p
    res = pd.DataFrame(res).groupby("predictor").agg(acc=("acc", "mean"), bacc=("bacc", "mean")).reset_index()
    for _, r in res.iterrows():
        rows.append(dict(base, predictor=r.predictor, acc=r.acc, bacc=r.bacc, n_folds=k,
                         majority=float(max(np.mean(yy == "leak"), np.mean(yy == "noleak")))))
    for name in EMB:
        d = sfb.paired_cluster_bootstrap_metrics(yy, store[name], store["clock (hour stump)"], gg,
                                                 n_boot=cfg.N_BOOT)
        diffs.append(dict(header=h, comparison=f"{name} - clock", **d))

wf = pd.DataFrame(rows)
wf.to_csv(cfg.RESULTS / "S_F_D2_window_by_format.csv", index=False)
print()
print(wf.round(4).to_string(index=False))
if diffs:
    wd = pd.DataFrame(diffs)
    wd.to_csv(cfg.RESULTS / "S_F_D2_window_by_format_differences.csv", index=False)
    print()
    print(wd[["header", "comparison", "n_clusters", "d_bacc", "d_bacc_lo", "d_bacc_hi", "d_acc", "d_acc_lo", "d_acc_hi"]]
          .round(4).to_string(index=False))
    for _, r in wd.iterrows():
        rp.record(f"D2.early_window.format[{r.header}].{r.comparison}.d_bacc", r.d_bacc,
                  "S_F_D2_window_by_format_differences.csv", "R2-6", lo=r.d_bacc_lo, hi=r.d_bacc_hi,
                  note=f"unit=logger, n_loggers={r.n_clusters}")
for _, r in wf[wf.predictor != "-"].iterrows():
    rp.record(f"D2.early_window.format[{r.header}].{r.predictor}.bacc", r.bacc, "S_F_D2_window_by_format.csv",
              "R2-6", note=f"n={r.n} ({r.n_leak} leak / {r.n_noleak} no-leak; loggers {r.leak_loggers}/{r.noleak_loggers}); acc={r.acc:.4f}")


files per recording format, class and sensor:
label                        leak  noleak
header        sensor                     
4096 Hz, 10 s Hydrophones      28       0
              Noise Loggers     4       2
8092 Hz, 10 s Hydrophones      12      20
              Noise Loggers     4       0

loggers per recording format and class:
label          leak  noleak
header                     
4096 Hz, 10 s     9       1
8092 Hz, 10 s     5       4

embeddings found: ['Mel2023', 'WAV2VEC2_BASE', 'HUBERT_BASE', 'AST']

       header  n  n_leak  n_noleak  leak_loggers  noleak_loggers          predictor                                                     note    acc   bacc  n_folds  majority
4096 Hz, 10 s 34      32         2             9               1                  - a class has fewer than 2 loggers: no logger-grouped test    NaN    NaN      NaN       NaN
8092 Hz, 10 s 36      16        20             5               4                AST                                               